In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [8]:
df = pd.read_csv('output.csv')
print(df.head())
df.drop(['task_type_1', 'task_type_2'], axis=1, inplace=True)
X=df.drop(['label','id'], axis=1)
y=df['label']


   id  token  label     task_type_1 task_type_2  task_type_prob  \
0   0     35      1  Classification         NaN           0.815   
1   1     54      1  Classification         NaN           0.819   
2   2     66      1  Classification   Closed QA           0.804   
3   3     68      1  Classification         NaN           0.799   
4   4     37      0  Classification         NaN           0.814   

   creativity_scope  reasoning  contextual_knowledge  number_of_few_shots  \
0            0.0030     0.0598                0.1112               0.0540   
1            0.0034     0.0240                0.0985               0.0654   
2            0.0031     0.0405                0.1278               0.1095   
3            0.0032     0.0325                0.0823               0.2283   
4            0.0040     0.0334                0.1158               0.0509   

   domain_knowledge  constraint_ct  prompt_complexity_score  
0            0.5401         0.8735                  0.23630  
1         

# Exploratory Data Analysis - Visualizing Y with X

In [3]:
# Distribution of target variable
import matplotlib as plt
plt.figure(figsize=(8, 6))
y.value_counts().plot(kind='bar', color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
plt.title('Distribution of Target Variable (Label)', fontsize=14, fontweight='bold')
plt.xlabel('Label', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Box plots for each feature grouped by label
feature_cols = X.columns
n_features = len(feature_cols)
n_cols = 3
n_rows = (n_features + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4))
axes = axes.flatten() if n_features > 1 else [axes]

for idx, col in enumerate(feature_cols):
    # Create a temporary DataFrame for plotting
    temp_df = pd.DataFrame({col: X[col], 'label': y})
    
    # Box plot
    temp_df.boxplot(column=col, by='label', ax=axes[idx])
    axes[idx].set_title(f'{col} by Label', fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Label', fontsize=10)
    axes[idx].set_ylabel(col, fontsize=10)
    plt.sca(axes[idx])
    plt.xticks(rotation=0)

# Remove extra subplots
for idx in range(n_features, len(axes)):
    fig.delaxes(axes[idx])

plt.suptitle('Feature Distributions by Label', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

# Violin plots for key features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

# Select top 4 most important features for violin plots
top_features = ['token', 'task_type_prob', 'reasoning', 'domain_knowledge']

for idx, col in enumerate(top_features[:4]):
    if col in X.columns:
        temp_df = pd.DataFrame({col: X[col], 'label': y})
        sns.violinplot(data=temp_df, x='label', y=col, ax=axes[idx], palette='Set2')
        axes[idx].set_title(f'{col} Distribution by Label', fontsize=12, fontweight='bold')
        axes[idx].set_xlabel('Label', fontsize=11)
        axes[idx].set_ylabel(col, fontsize=11)

plt.suptitle('Violin Plots: Feature Distributions by Label', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Correlation heatmap with target
plt.figure(figsize=(10, 8))
correlation_df = pd.concat([X, y], axis=1)
correlation_matrix = correlation_df.corr()

sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', 
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix (Features and Target)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Pair plot for selected features (if not too many)
print("\nGenerating pair plot for selected features...")
selected_features = ['token', 'reasoning', 'domain_knowledge', 'prompt_complexity_score']
pair_df = pd.DataFrame(X[selected_features])
pair_df['label'] = y

sns.pairplot(pair_df, hue='label', palette='Set1', diag_kind='kde', height=2.5)
plt.suptitle('Pair Plot: Selected Features by Label', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

TypeError: 'module' object is not callable

feature scaling

In [9]:
X.count

<bound method DataFrame.count of        token  task_type_prob  creativity_scope  reasoning  \
0         35           0.815            0.0030     0.0598   
1         54           0.819            0.0034     0.0240   
2         66           0.804            0.0031     0.0405   
3         68           0.799            0.0032     0.0325   
4         37           0.814            0.0040     0.0334   
...      ...             ...               ...        ...   
12287    107           0.901            0.0041     0.0169   
12288    107           0.916            0.0043     0.0165   
12289    110           0.912            0.0039     0.0179   
12290    116           0.840            0.0032     0.0135   
12291    114           0.854            0.0025     0.0134   

       contextual_knowledge  number_of_few_shots  domain_knowledge  \
0                    0.1112               0.0540            0.5401   
1                    0.0985               0.0654            0.8011   
2                    0.1

In [10]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X = scaler.fit_transform(X)

label encoding(ig not required here)

In [9]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(y)

split data

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [12]:
pd.Series(y_train).value_counts(normalize=True)

label
1    0.649853
0    0.350147
Name: proportion, dtype: float64

logistic regression

In [13]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    solver="lbfgs",             # supports multinomial
    max_iter=5000
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_pred2=model.predict(X_train)
y_prob = model.predict_proba(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f" Test Accuracy: {accuracy:.4f}")
accuracy = accuracy_score(y_train, y_pred2)
print(f" Train Accuracy: {accuracy:.4f}")
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))


 Test Accuracy: 0.7926
 Train Accuracy: 0.7900
              precision    recall  f1-score   support

           0       0.83      0.51      0.63       861
           1       0.78      0.94      0.86      1598

    accuracy                           0.79      2459
   macro avg       0.81      0.73      0.74      2459
weighted avg       0.80      0.79      0.78      2459



/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/linear_model/_linear_loss.py:333: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/linear_model/_linea

# Random Forest with Hyperparameter Tuning

In [14]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    bootstrap=False, class_weight=None, max_depth=10, min_samples_leaf=3, min_samples_split=13, n_estimators=100
)

rf.fit(X_train, y_train)
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))
y_pred = rf.predict(X_test)
y_pred2=rf.predict(X_train)
y_prob = rf.predict_proba(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f" Test Accuracy: {accuracy:.4f}")
accuracy = accuracy_score(y_train, y_pred2)
print(f" Train Accuracy: {accuracy:.4f}")

              precision    recall  f1-score   support

           0       0.83      0.51      0.63       861
           1       0.78      0.94      0.86      1598

    accuracy                           0.79      2459
   macro avg       0.81      0.73      0.74      2459
weighted avg       0.80      0.79      0.78      2459

 Test Accuracy: 0.7950
 Train Accuracy: 0.8077


## Hyperparameter Tuning using RandomizedSearchCV

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform
import time

# Define parameter distributions for RandomizedSearchCV
param_distributions = {
    'n_estimators': randint(50, 300),           # Number of trees
    'max_depth': [None, 5, 10, 15, 20, 25],     # Maximum depth of trees
    'min_samples_split': randint(2, 20),        # Minimum samples to split node
    'min_samples_leaf': randint(1, 10),         # Minimum samples in leaf
    'max_features': ['sqrt', 'log2', None],     # Features to consider for split
    'bootstrap': [True, False],                  # Bootstrap samples
    'class_weight': ['balanced', 'balanced_subsample', None]
}

# Initialize base model
rf_base = RandomForestClassifier(random_state=42)

# Initialize RandomizedSearchCV
print("Starting hyperparameter tuning with RandomizedSearchCV...")
print("This may take a few minutes...\n")

start_time = time.time()

random_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_distributions,
    n_iter=50,                    # Number of parameter combinations to try
    cv=5,                         # 5-fold cross-validation
    scoring='accuracy',
    n_jobs=-1,                    # Use all processors
    verbose=2,
    random_state=42,
    return_train_score=True
)

# Fit the random search
random_search.fit(X_train, y_train)

end_time = time.time()

print(f"\n{'='*60}")
print("Hyperparameter Tuning Complete!")
print(f"Time taken: {end_time - start_time:.2f} seconds")
print(f"{'='*60}\n")

# Best parameters and score
print("Best Parameters:")
for param, value in random_search.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nBest Cross-Validation Accuracy: {random_search.best_score_:.4f}")

# Use best model for predictions
best_rf = random_search.best_estimator_
y_pred_tuned = best_rf.predict(X_test)
y_pred_train_tuned = best_rf.predict(X_train)

test_accuracy_tuned = accuracy_score(y_test, y_pred_tuned)
train_accuracy_tuned = accuracy_score(y_train, y_pred_train_tuned)

print(f"\nTest Accuracy (Tuned): {test_accuracy_tuned:.4f}")
print(f"Train Accuracy (Tuned): {train_accuracy_tuned:.4f}")
print(f"\nClassification Report (Tuned Model):")
print(classification_report(y_test, y_pred_tuned))

Starting hyperparameter tuning with RandomizedSearchCV...
This may take a few minutes...

Fitting 5 folds for each of 50 candidates, totalling 250 fits
[CV] END bootstrap=True, class_weight=balanced, max_depth=10, max_features=sqrt, min_samples_leaf=5, min_samples_split=8, n_estimators=171; total time=   0.7s
[CV] END bootstrap=True, class_weight=balanced, max_depth=10, max_features=sqrt, min_samples_leaf=5, min_samples_split=8, n_estimators=171; total time=   0.7s
[CV] END bootstrap=True, class_weight=balanced, max_depth=10, max_features=sqrt, min_samples_leaf=5, min_samples_split=8, n_estimators=171; total time=   0.8s
[CV] END bootstrap=True, class_weight=balanced, max_depth=10, max_features=sqrt, min_samples_leaf=5, min_samples_split=8, n_estimators=171; total time=   0.7s
[CV] END bootstrap=True, class_weight=balanced, max_depth=10, max_features=sqrt, min_samples_leaf=5, min_samples_split=8, n_estimators=171; total time=   0.7s
[CV] END bootstrap=True, class_weight=None, max_depth

## Model Performance Comparison

In [ ]:
# Compare results from base model vs. tuned model
comparison_df = pd.DataFrame({
    'Model': ['Base RF', 'Tuned RF'],
    'Train Accuracy': [accuracy_score(y_train, y_pred2), train_accuracy_tuned],
    'Test Accuracy': [accuracy_score(y_test, y_pred), test_accuracy_tuned]
})

print("\n" + "="*60)
print("Model Performance Comparison")
print("="*60)
print(comparison_df.to_string(index=False))
print("="*60 + "\n")

# Visualize the comparison
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart comparison
x = np.arange(len(comparison_df))
width = 0.35

ax[0].bar(x - width/2, comparison_df['Train Accuracy'], width, label='Train Accuracy', color='#4ECDC4')
ax[0].bar(x + width/2, comparison_df['Test Accuracy'], width, label='Test Accuracy', color='#FF6B6B')
ax[0].set_xlabel('Model', fontsize=12, fontweight='bold')
ax[0].set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax[0].set_title('Model Accuracy Comparison', fontsize=14, fontweight='bold')
ax[0].set_xticks(x)
ax[0].set_xticklabels(comparison_df['Model'])
ax[0].legend()
ax[0].grid(axis='y', alpha=0.3)
ax[0].set_ylim([0, 1])

# Confusion Matrix for Tuned Model
from sklearn.metrics import confusion_matrix
cm_tuned = confusion_matrix(y_test, y_pred_tuned)
sns.heatmap(cm_tuned, annot=True, fmt='d', cmap='Blues', ax=ax[1], 
            xticklabels=['Class 0', 'Class 1', 'Class 2'],
            yticklabels=['Class 0', 'Class 1', 'Class 2'])
ax[1].set_title('Confusion Matrix - Tuned Model', fontsize=14, fontweight='bold')
ax[1].set_ylabel('True Label', fontsize=12, fontweight='bold')
ax[1].set_xlabel('Predicted Label', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

# Feature importance from tuned model
feature_names = df.drop(['label','id'], axis=1).columns
feature_importance = best_rf.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importance
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance_df['Feature'], feature_importance_df['Importance'], color='#45B7D1')
plt.xlabel('Importance', fontsize=12, fontweight='bold')
plt.ylabel('Feature', fontsize=12, fontweight='bold')
plt.title('Feature Importance - Tuned Random Forest', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("\nTop 5 Most Important Features:")
print(feature_importance_df.head().to_string(index=False))

## Cross-Validation Results Analysis

In [ ]:
# Extract and visualize cross-validation results
cv_results = pd.DataFrame(random_search.cv_results_)

# Sort by test score
cv_results_sorted = cv_results.sort_values(by='rank_test_score').head(10)

print("Top 10 Parameter Combinations:")
print(cv_results_sorted[['params', 'mean_test_score', 'std_test_score', 'rank_test_score']].to_string(index=False))

# Visualize parameter importance
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# n_estimators vs accuracy
axes[0, 0].scatter(cv_results['param_n_estimators'], cv_results['mean_test_score'], alpha=0.6, color='#4ECDC4')
axes[0, 0].set_xlabel('n_estimators', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Mean CV Accuracy', fontsize=11, fontweight='bold')
axes[0, 0].set_title('n_estimators vs Accuracy', fontsize=12, fontweight='bold')
axes[0, 0].grid(alpha=0.3)

# max_depth vs accuracy
max_depth_values = [str(x) if x is not None else 'None' for x in cv_results['param_max_depth']]
axes[0, 1].scatter(range(len(max_depth_values)), cv_results['mean_test_score'], alpha=0.6, color='#FF6B6B')
axes[0, 1].set_xlabel('max_depth', fontsize=11, fontweight='bold')
axes[0, 1].set_ylabel('Mean CV Accuracy', fontsize=11, fontweight='bold')
axes[0, 1].set_title('max_depth vs Accuracy', fontsize=12, fontweight='bold')
axes[0, 1].grid(alpha=0.3)

# min_samples_split vs accuracy
axes[1, 0].scatter(cv_results['param_min_samples_split'], cv_results['mean_test_score'], alpha=0.6, color='#45B7D1')
axes[1, 0].set_xlabel('min_samples_split', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Mean CV Accuracy', fontsize=11, fontweight='bold')
axes[1, 0].set_title('min_samples_split vs Accuracy', fontsize=12, fontweight='bold')
axes[1, 0].grid(alpha=0.3)

# Distribution of CV scores
axes[1, 1].hist(cv_results['mean_test_score'], bins=20, color='#95E1D3', edgecolor='black')
axes[1, 1].axvline(random_search.best_score_, color='red', linestyle='--', linewidth=2, label=f'Best Score: {random_search.best_score_:.4f}')
axes[1, 1].set_xlabel('Mean CV Accuracy', fontsize=11, fontweight='bold')
axes[1, 1].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[1, 1].set_title('Distribution of CV Scores', fontsize=12, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.suptitle('Hyperparameter Impact on Model Performance', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [19]:
# After: df = pd.read_csv('advanced_training_data.csv')
print(f"\n=== Duplicate Check ===")
print(f"Total rows: {len(df)}")
print(f"Duplicate rows: {df.duplicated().sum()}")

# Show duplicate rows if any
if df.duplicated().sum() > 0:
    print("\nDuplicate rows found:")
    print(df[df.duplicated(keep=False)].sort_values(by=list(df.columns)))


=== Duplicate Check ===
Total rows: 7168
Duplicate rows: 462

Duplicate rows found:
       id  token  label  task_type_prob  creativity_scope  reasoning  \
1872    0    141      0           0.928            0.0034     0.0111   
2334    0    141      0           0.928            0.0034     0.0111   
1873    1    141      0           0.926            0.0030     0.0115   
2335    1    141      0           0.926            0.0030     0.0115   
1874    2    142      0           0.923            0.0032     0.0117   
...   ...    ...    ...             ...               ...        ...   
2793  459     74      0           0.920            0.0033     0.0402   
2332  460    135      1           0.958            0.0031     0.0232   
2794  460    135      1           0.958            0.0031     0.0232   
2333  461    135      0           0.967            0.0028     0.0291   
2795  461    135      0           0.967            0.0028     0.0291   

      contextual_knowledge  number_of_few_shots  d